In [103]:
import numpy as np
import torch
import torch.nn as nn
import h5py
import matplotlib.pyplot as plt
plt.style.use("styles/plots.mplstyle")

In [ ]:
class LearnedTikhonov(nn.Module):
    """
    Filtro W_θ(ω) = conj(H) * exp(iφ_θ) / (|H|² + R_θ)

    Buffers (fissi):
      H_abs2  : |H(ω)|²          (nk,) float32
      H_conj  : conj(H(ω))       (nk,) complex64

    Parametri appresi:
      phi_raw : φ_θ(ω)  ∈ ℝ^nk
      log_R   : log R_θ(ω) ∈ ℝ^nk  →  R = exp(log_R) > 0
    """

    def __init__(self, H: np.ndarray, n: int, nt: int, dt: float):
        super().__init__()
        self.n  = n
        self.nt = nt
        self.dt = dt
        nk = len(H)
        self.nk = nk
        self.n_even = (n % 2 == 0)

        H_t = torch.from_numpy(H.astype(np.complex64))
        self.register_buffer("H_abs2", H_t.abs().pow(2))   # float32
        self.register_buffer("H_conj", H_t.conj())          # complex64

        self.phi_raw = nn.Parameter(torch.zeros(nk))
        self.log_R   = nn.Parameter(torch.full((nk,), np.log(1e-2)))

    def _get_phi(self) -> torch.Tensor:
        """Forza φ[0]=0 (DC). Se n pari forza anche φ[-1]=0 (Nyquist)."""
        phi = self.phi_raw.clone()
        phi[0] = 0.0
        if self.n_even:
            phi[-1] = 0.0
        return phi                           # (nk,) float32, stesso device

    def _get_R(self) -> torch.Tensor:
        return torch.exp(self.log_R)         # (nk,) float32, sempre > 0

    def _build_W(self) -> torch.Tensor:
        """Costruisce il filtro W_θ (nk,) complex64."""
        phi = self._get_phi()
        R   = self._get_R()
        # torch.polar: device-safe, niente letterali Python complessi
        exp_i_phi = torch.polar(torch.ones_like(phi), phi)  # (nk,) complex64
        return self.H_conj * exp_i_phi / (self.H_abs2 + R + 1e-12)

    def forward(self, g: torch.Tensor) -> torch.Tensor:
        """
        g : (N, nt) float32  — segnale al bordo sinistro
        Ritorna f_pred : (N, nt) float32
        """
        # Integrazione trapezoidale
        g_trap = g.clone()
        g_trap[:, :-1] = (g[:, :-1] + g[:, 1:]) / 2.0
        g_trap[:,  -1] = g[:, -1] / 2.0

        G      = torch.fft.rfft(g_trap, n=self.n, dim=1)   # (N, nk)
        W      = self._build_W()                             # (nk,)
        F_pred = G * W.unsqueeze(0)                          # (N, nk)

        f_pred = torch.fft.irfft(F_pred, n=self.n, dim=1)[:, :self.nt]
        return f_pred / self.dt

    # ------------------------------------------------------------------
    # Utilità diagnostiche
    # ------------------------------------------------------------------
    @torch.no_grad()
    def get_phi(self) -> np.ndarray:
        return self._get_phi().cpu().numpy()

    @torch.no_grad()
    def get_R(self) -> np.ndarray:
        return self._get_R().cpu().numpy()

    @torch.no_grad()
    def get_W_magnitude(self) -> np.ndarray:
        return self._build_W().abs().cpu().numpy()

In [105]:
def smoothness_regularization(model: LearnedTikhonov) -> torch.Tensor:
    """
    Penalizza variazioni brusche di R(omega) e Phi(omega).
    Assumendo che model.R e model.Phi siano nn.Parameter di shape (N_omega,).
    """
    dR   = model._get_R()[1:]   - model._get_R()[:-1]      # differenze finite del 1° ordine
    dPhi = model._get_phi()[1:] - model._get_phi()[:-1]
    return (dR ** 2).mean() + (dPhi ** 2).mean()


def train(
    model: LearnedTikhonov,
    G_train: torch.Tensor,
    F_train: torch.Tensor,
    G_val:   torch.Tensor,
    F_val:   torch.Tensor,
    n_epochs: int        = 2000,
    lr: float            = 1e-3,
    weight_decay: float  = 0.0,
    checkpoint_path: str = "assets/tikhonov_learned/best_tikhonov.pt",
    scheduler_patience: int   = 50,
    scheduler_factor:   float = 0.5,
    log_every: int       = 50,
) -> dict:

    optimizer = torch.optim.Adam(
        model.parameters(), lr=lr, weight_decay=weight_decay
    )
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="min", patience=scheduler_patience,
        factor=scheduler_factor
    )
    
    loss_fn = nn.MSELoss()

    history = {"train_loss": [], "val_loss": [], "lr": []}
    best_val = float("inf")

    for epoch in range(1, n_epochs + 1):

        # ---- train: un unico passo su tutti i dati ----
        model.train()
        optimizer.zero_grad()
        
        lambda_smooth: float = 1e-4 
        
        pred_train = model(G_train)
        mse_train = loss_fn(pred_train, F_train)
        reg_smooth  = smoothness_regularization(model) # termine che impone la continuità di phi ed R
        loss_train  = mse_train + lambda_smooth * reg_smooth
        
        loss_train.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        # ---- validation ----
        model.eval()
        with torch.no_grad():
            pred_val = model(G_val)
            mse_val = loss_fn(pred_val, F_val)
            reg_smooth  = smoothness_regularization(model) # termine che impone la continuità di phi ed R
            loss_val  = mse_val + lambda_smooth * reg_smooth
        
            #loss_val = loss_fn(model(G_val), F_val)

        train_l = loss_train.item()
        val_l   = loss_val.item()
        scheduler.step(val_l)
        current_lr = optimizer.param_groups[0]["lr"]

        history["train_loss"].append(train_l)
        history["val_loss"].append(val_l)
        history["lr"].append(current_lr)

        if val_l < best_val:
            best_val = val_l
            torch.save({
                "epoch": epoch,
                "model_state": model.state_dict(),
                "optimizer_state": optimizer.state_dict(),
                "val_loss": best_val,
            }, checkpoint_path)

        if epoch % log_every == 0 or epoch == 1:
            print(f"Epoch {epoch:5d}/{n_epochs} | "
                  f"train MSE: {train_l:.4e} | "
                  f"val MSE: {val_l:.4e} | "
                  f"lr: {current_lr:.2e}")

    print(f"\nBest val MSE: {best_val:.4e}  →  {checkpoint_path}")
    return history

In [ ]:
def plot_diagnostics(model: LearnedTikhonov, freqs: np.ndarray,
                     save_path: str = "diagnostics.png"):
    fig, axes = plt.subplots(3, 1, figsize=(10, 9), sharex=True)

    axes[0].plot(freqs, np.degrees(model.get_phi()), color="steelblue", lw=1.5)
    axes[0].axhline(0, color="k", lw=0.5, ls="--")
    axes[0].set_ylabel("Phi")
    axes[0].set_title("Fase appresa")
    axes[0].grid(True, alpha=0.3)

    axes[1].semilogy(freqs, model.get_R(), color="darkorange", lw=1.5)
    axes[1].set_ylabel("R")
    axes[1].set_title("Prior di regolarizzazione")
    axes[1].grid(True, alpha=0.3)

    axes[2].semilogy(freqs, model.get_W_magnitude(), color="seagreen", lw=1.5)
    axes[2].set_ylabel("|W|")
    axes[2].set_xlabel("Frequenza [Hz]")
    axes[2].set_title("Magnitudine del filtro WW")
    axes[2].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.close()
    print(f"Diagnostica salvata → {save_path}")


def plot_history(history: dict, save_path: str = "loss_history.png"):
    epochs = range(1, len(history["train_loss"]) + 1)
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.semilogy(epochs, history["train_loss"], label="Train MSE")
    ax.semilogy(epochs, history["val_loss"],   label="Val MSE", ls="--")
    ax.set_xlabel("Epoca")
    ax.set_ylabel("MSE")
    ax.set_title("Curva di apprendimento")
    ax.legend()
    ax.grid(True, alpha=0.3)
    #plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.close()
    print(f"Loss history salvata → {save_path}")

def plot_prediction(model: LearnedTikhonov,
                    G_val: torch.Tensor,
                    F_val: torch.Tensor,
                    dt: float,
                    n_samples: int = 3,
                    save_path: str = "predictions.png"):
    """Confronta f_pred vs f_true su n_samples campioni di validazione."""
    model.eval()
    with torch.no_grad():
        f_pred = model(G_val).cpu().numpy()
    f_true = F_val.cpu().numpy()
    nt = f_true.shape[1]
    t  = np.arange(nt) * dt

    n_show = min(n_samples, len(f_true))
    fig, axes = plt.subplots(n_show, 1, figsize=(10, 3 * n_show), sharex=True)
    if n_show == 1:
        axes = [axes]

    for i, ax in enumerate(axes):
        ax.plot(t, f_true[i], label="f true", lw=1.5)
        ax.plot(t, f_pred[i], label="f pred", lw=1.5, ls="--")
        mse_i = np.mean((f_pred[i] - f_true[i]) ** 2)
        ax.set_title(f"Campione val {i}  —  MSE = {mse_i:.3e}")
        ax.set_ylabel("f(t)")
        ax.legend(loc="upper right")
        ax.grid(True, alpha=0.3)

    axes[-1].set_xlabel("Tempo [s]")
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.close()
    print(f"Predizioni salvate → {save_path}")


In [ ]:
# -----------------------------------------------------------------------
# 1. Parametri simulazione
# -----------------------------------------------------------------------
L, Tf, nt, D, h_param, l, _, r, _ = np.loadtxt(
    "dataset/simulation_info.csv", delimiter=";", usecols=1
)
nt = int(nt)
dt = Tf / nt
n  = int(2 * nt - 1)

h_vals = np.loadtxt("dataset/impulse_response.csv", delimiter=";")
H_np   = np.fft.rfft(h_vals, n).astype(np.complex64)
nk     = len(H_np)
freqs  = np.fft.rfftfreq(n, d=dt)

print(f"nt={nt}, n={n}, nk={nk}, dt={dt:.4e}")

# -----------------------------------------------------------------------
# 2. Caricamento dataset completo in memoria
# -----------------------------------------------------------------------
VAL_FRACTION = 0.1
SEED         = 42
H5_PATH      = "dataset/dataset.h5"

with h5py.File(H5_PATH, "r") as hf:
    data = hf["dataset"][:]              # (N, 2, nt)

f_all = data[:, 0, :].astype(np.float32)
g_all = data[:, 1, :].astype(np.float32)

rolling = 0 # Simulate a delay of: tau = rolling * dt
if rolling != 0:
    g_rolled = np.roll(g_all, rolling)
    g_rolled[:,rolling:] = 0
    g_all = g_rolled

N = len(f_all)

rng       = np.random.default_rng(SEED)
idx       = rng.permutation(N)
n_val     = max(1, int(VAL_FRACTION * N))
idx_val   = idx[:n_val]
idx_train = idx[n_val:]

print(f"Campioni: totali={N}, train={len(idx_train)}, val={n_val}")

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Dispositivo: {device}")

# Tutti i dati sul device — nessun DataLoader
G_train = torch.from_numpy(g_all[idx_train]).to(device)
F_train = torch.from_numpy(f_all[idx_train]).to(device)
G_val   = torch.from_numpy(g_all[idx_val]).to(device)
F_val   = torch.from_numpy(f_all[idx_val]).to(device)

# -----------------------------------------------------------------------
# 3. Modello
# -----------------------------------------------------------------------
model = LearnedTikhonov(H=H_np, n=n, nt=nt, dt=dt).to(device)
n_params = sum(p.numel() for p in model.parameters())
print(f"Parametri appresi: {n_params}  (phi: {nk}, log_R: {nk})")

nt=2000, n=3999, nk=2000, dt=5.0000e-03
Campioni: totali=500, train=450, val=50
Dispositivo: cuda
Parametri appresi: 4000  (phi: 2000, log_R: 2000)


# 4. Training

In [108]:
history = train(
    model        = model,
    G_train      = G_train,
    F_train      = F_train,
    G_val        = G_val,
    F_val        = F_val,
    n_epochs     = 9000,
    lr           = 1e-3,
    weight_decay = 0.0,
    checkpoint_path    = "assets/tikhonov_learned/best_tikhonov.pt",
    scheduler_patience = 50,
    scheduler_factor   = 0.5,
    log_every    = 100,
)

Epoch     1/9000 | train MSE: 2.8883e+01 | val MSE: 3.2517e+01 | lr: 1.00e-03
Epoch   100/9000 | train MSE: 2.5067e+01 | val MSE: 2.8371e+01 | lr: 1.00e-03
Epoch   200/9000 | train MSE: 2.1231e+01 | val MSE: 2.4172e+01 | lr: 1.00e-03
Epoch   300/9000 | train MSE: 1.7536e+01 | val MSE: 2.0094e+01 | lr: 1.00e-03
Epoch   400/9000 | train MSE: 1.4094e+01 | val MSE: 1.6262e+01 | lr: 1.00e-03
Epoch   500/9000 | train MSE: 1.0991e+01 | val MSE: 1.2775e+01 | lr: 1.00e-03
Epoch   600/9000 | train MSE: 8.2934e+00 | val MSE: 9.7144e+00 | lr: 1.00e-03
Epoch   700/9000 | train MSE: 6.0393e+00 | val MSE: 7.1291e+00 | lr: 1.00e-03
Epoch   800/9000 | train MSE: 4.3328e+00 | val MSE: 5.1510e+00 | lr: 1.00e-03
Epoch   900/9000 | train MSE: 3.1979e+00 | val MSE: 3.8187e+00 | lr: 1.00e-03
Epoch  1000/9000 | train MSE: 2.4329e+00 | val MSE: 2.9092e+00 | lr: 1.00e-03
Epoch  1100/9000 | train MSE: 1.9092e+00 | val MSE: 2.2794e+00 | lr: 1.00e-03
Epoch  1200/9000 | train MSE: 1.5447e+00 | val MSE: 1.8367e+00 |

# 5. Diagnostica

In [ ]:
ckpt = torch.load("assets/tikhonov_learned/best_tikhonov.pt", map_location="cpu")
model.load_state_dict(ckpt["model_state"])
model.to("cpu")

plot_diagnostics(model, freqs,              save_path="assets/tikhonov_learned/diagnostics.png")
plot_history(history,                        save_path="assets/tikhonov_learned/loss_history.png")
plot_prediction(model, G_val.cpu(), F_val.cpu(), dt,
                n_samples=min(20, n_val),     save_path="assets/tikhonov_learned/predictions.png")

# Dataset misto in fase di validazione e training

/tmp/ipykernel_36785/4228262232.py:22: UserWarning: The figure layout has changed to tight
  plt.tight_layout()


Diagnostica salvata → assets/tikhonov_learned/diagnostics.png
Loss history salvata → assets/tikhonov_learned/loss_history.png


/tmp/ipykernel_36785/4228262232.py:73: UserWarning: The figure layout has changed to tight
  plt.tight_layout()


Predizioni salvate → assets/tikhonov_learned/predictions.png
